<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_4_normalizing_flows_density_estimation_direct_likelihood.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 4 — Direct density estimation with normalizing flows

In the previous exercises we estimated density ratios directly with classifiers.  Here we do a complementary exercise: train two normalizing flows,

$$
\hat p_\mathrm{sig}(x), \qquad \hat p_\mathrm{bkg}(x),
$$

where $x=(x_1,\ldots,x_5)$ are the reconstructed/smeared features.  We then:

1. choose either an affine-coupling RealNVP or a rational-quadratic-spline flow, and train one flow for the background and one for the signal;
2. validate the learned densities using MC feature projections, the distribution of the learned log density, and, for this toy problem, the analytic smeared Gaussian-mixture truth;
3. build the likelihood **directly from the densities**, without forming density ratios and without constructing a workspace;
4. fit the signal-strength parameter $\mu$ with a small JAX likelihood;
5. use the explicit toy density to generate pseudo-experiments and visualize the distribution of the likelihood-ratio test statistic.

The main pedagogical contrast is:

- classifier method: learn $p_i(x)/p_\mathrm{ref}(x)$ directly;
- flow method: learn $p_i(x)$ directly and insert those densities in the likelihood.

In [1]:
## ============================================================================
# Google Colab setup — run me first.  Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"  # package + tutorial helpers
BRANCH   = "ml4hep_school_tutorial"
N_BKG, N_SIG = 2_000_000, 2_000_000     # Colab-sized dataset (raise for less MC noise in the fit)
USE_DRIVE = True                    # True -> save data/models to Google Drive so they
                                    # persist across notebooks & sessions (see notes above)
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab"
    else:
        ROOT = "/content"
    os.makedirs(ROOT, exist_ok=True)
    os.chdir(ROOT)

    # 1) fetch ONLY the package source + tutorial helpers (skip Git-LFS / big blobs)
    if not os.path.isdir("nsbi-lhc-toolkit"):
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        !git clone --depth 1 --filter=blob:none --sparse --branch $BRANCH $REPO_URL
        !cd nsbi-lhc-toolkit && git sparse-checkout set src workshops/ml4hep_tifr

    # 2) make `import nsbi_common_utils` work (pure-python src layout, no build step)
    src = os.path.abspath("nsbi-lhc-toolkit/src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # 3) runtime deps Colab doesn't already ship (torch/jax/sklearn/... are preinstalled)
    !pip install -q pytorch-lightning onnx onnxruntime onnxscript iminuit mplhep nflows

    # 4) work from the tutorial dir so utils.py / generate_distributions.py and the
    #    ./dataframes, ./models_* relative paths resolve just like a local run
    os.chdir("nsbi-lhc-toolkit/workshops/ml4hep_tifr")

    # 5) generate the Gaussian-mixture samples if they aren't there yet
    if not os.path.exists("dataframes/signal.parquet"):
        !python generate_distributions.py --n_bkg $N_BKG --n_sig $N_SIG

print("Working dir:", os.getcwd())

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 128.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 448.2/448.2 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 159.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 161.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/98

## Inputs expected by this notebook

This notebook expects the updated generator output:

```text
dataframes/background.parquet
dataframes/signal.parquet
```

with both truth-level columns `z1,...,z5` and reco-level columns `x1,...,x5`.  The fit below uses only the reco-level `x*` variables.

In [ ]:
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

import torch

import nsbi_common_utils
from nsbi_common_utils.training import density_ratio_trainer, predict_with_model

from utils import (
    FEATURES,
    background_components,
    signal_components,
    smearing_parameters,
    split_train_inference,
)
from utils_nf import flow_log_prob_x, flow_sample_x, train_flow
from utils_plotting import (
    plot_flow_pair_closure,
    plot_log_density_truth_binned,
    plot_log_density_truth_scatter,
    plot_log_prob_cdf_closure,
    plot_log_prob_closure,
    plot_mu_hat_toys,
    plot_profile_scan,
    plot_profile_scan_comparison,
    plot_t_mu_toys,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)

SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")

In [ ]:
BASE_PATH = Path("./dataframes")
PRESEL_MODEL_DIR = Path("models_PRESEL")
PRESEL_PLOT_DIR = Path("plots_PRESEL")
FLOW_MODEL_DIR = Path("models_flows_preselected")
DENSITY_DIR = Path("saved_densities_flows_preselected")
PLOT_DIR = Path("plots_flows_preselected")

for directory in [
    PRESEL_MODEL_DIR,
    PRESEL_PLOT_DIR,
    FLOW_MODEL_DIR,
    DENSITY_DIR,
    PLOT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Use independent events to train the preselection, train the flows, and
# evaluate the final likelihood. split_train_inference rescales each subset's
# weights to the original physical yield.
SPLIT_SEED = 0
PRESEL_TRAIN_FRACTION = 0.5
FLOW_TRAIN_FRACTION = 0.5  # fraction of the remaining events used by the flows

# Preselection classifier choices. The classifier estimates p_signal/p_background.
PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO = 100.0
PRESEL_MAX_TRAIN_EVENTS_PER_CLASS = 500_000
PRESEL_HIDDEN_LAYERS = 3
PRESEL_NEURONS = 256
PRESEL_N_EPOCHS = 30
PRESEL_BATCH_SIZE = 2048
PRESEL_LEARNING_RATE = 1.0e-3
PRESEL_HOLDOUT_FRACTION = 0.25
PRESEL_VALIDATION_FRACTION = 0.20
PRESEL_PATIENCE = 8
PRESEL_LOAD_IF_AVAILABLE = True

# Select the discrete normalizing-flow architecture.
# False: affine-coupling RealNVP; True: rational-quadratic-spline couplings.
USE_QUADRATIC_SPLINE = False
FLOW_TYPE = "quadratic_spline" if USE_QUADRATIC_SPLINE else "realnvp"

# Keep these modest for a tutorial. Increase N_EPOCHS and MAX_TRAIN_EVENTS
# for tighter closure against the analytic truth.
MAX_TRAIN_EVENTS = {
    "background": 1_000_000,
    "signal": 1_000_000,
}

# Model architecture: these choices stay visible and editable for students.
N_COUPLING_LAYERS = 8
HIDDEN_FEATURES = 1024
HIDDEN_LAYERS = 4
SCALE_CLIP = 1.5                 # RealNVP only
SPLINE_NUM_BINS = 8              # quadratic-spline flow only
SPLINE_TAIL_BOUND = 3.0          # in standardized coordinates; spline only
DROPOUT_PROBABILITY = 0.0        # spline conditioner only

# Optimizer and training choices.
BATCH_SIZE = 2048
N_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
VALIDATION_FRACTION = 0.20
PATIENCE = 5
GRADIENT_CLIP = 5.0

# If checkpoints already exist, load the selected architecture instead of
# retraining. RealNVP and spline checkpoints have distinct filenames.
LOAD_IF_AVAILABLE = True

MODEL_CONFIG = {
    "flow_type": FLOW_TYPE,
    "n_features": N_DIM,
    "n_coupling_layers": N_COUPLING_LAYERS,
    "hidden_features": HIDDEN_FEATURES,
    "hidden_layers": HIDDEN_LAYERS,
    "scale_clip": SCALE_CLIP,
    "spline_num_bins": SPLINE_NUM_BINS,
    "spline_tail_bound": SPLINE_TAIL_BOUND,
    "dropout_probability": DROPOUT_PROBABILITY,
}

TRAINING_CONFIG = {
    "batch_size": BATCH_SIZE,
    "n_epochs": N_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "validation_fraction": VALIDATION_FRACTION,
    "patience": PATIENCE,
    "gradient_clip": GRADIENT_CLIP,
}

print(f"Selected flow architecture: {FLOW_TYPE}")

## Load data and define independent samples

The inclusive signal-to-background ratio is deliberately left at its physical, difficult value. We first reserve events for a **preselection classifier**. The remaining events are split again into a sample for choosing the cut and training the flows, and an independent sample for validation and the likelihood fit.

Keeping these roles disjoint prevents either classifier or flow overtraining from leaking directly into the final inference sample.

In [ ]:
signal = pd.read_parquet(BASE_PATH / "signal.parquet")
background = pd.read_parquet(BASE_PATH / "background.parquet")

missing = [f for f in FEATURES if f not in signal.columns or f not in background.columns]
if missing:
    raise RuntimeError(
        "Missing reco-level columns "
        f"{missing}. Regenerate the parquet files with the updated generator "
        "that writes x1,...,x5."
    )

for name, df in [("signal", signal), ("background", background)]:
    if "weight" not in df.columns:
        raise RuntimeError(f"{name} dataframe is missing a 'weight' column.")
    print(
        f"{name:10s}: {len(df):,} events, "
        f"sum weights = {df['weight'].sum():.6g}"
    )

In [ ]:
PRESEL_INCLUSIVE_YIELD = {
    "signal": float(signal["weight"].sum()),
    "background": float(background["weight"].sum()),
}

PRESEL_signal_train, signal_flow_pool = split_train_inference(
    signal,
    train_fraction=PRESEL_TRAIN_FRACTION,
    seed=SPLIT_SEED,
)
PRESEL_background_train, background_flow_pool = split_train_inference(
    background,
    train_fraction=PRESEL_TRAIN_FRACTION,
    seed=SPLIT_SEED,
)

signal_train_unselected, signal_eval_unselected = split_train_inference(
    signal_flow_pool,
    train_fraction=FLOW_TRAIN_FRACTION,
    seed=SPLIT_SEED + 1,
)
background_train_unselected, background_eval_unselected = split_train_inference(
    background_flow_pool,
    train_fraction=FLOW_TRAIN_FRACTION,
    seed=SPLIT_SEED + 1,
)

print(
    "Inclusive B/S = "
    f"{PRESEL_INCLUSIVE_YIELD['background'] / PRESEL_INCLUSIVE_YIELD['signal']:,.1f}"
)
print(
    f"PRESEL training: {len(PRESEL_signal_train):,} signal + "
    f"{len(PRESEL_background_train):,} background events"
)
print(
    f"Flow pool:       {len(signal_train_unselected):,} signal + "
    f"{len(background_train_unselected):,} background events"
)
print(
    f"Fit/eval pool:   {len(signal_eval_unselected):,} signal + "
    f"{len(background_eval_unselected):,} background events"
)

## Preselection with a learned density ratio

A binary classifier trained with equal total weight for signal and background estimates the density ratio

$$
r_{\rm PRESEL}(x) = \frac{p_{\rm signal}(x)}{p_{\rm background}(x)}
\simeq \frac{s(x)}{1-s(x)}.
$$

We use the repository's BCE density-ratio trainer. This first model is only a preselection tool; the final likelihood will still be constructed from the two normalizing-flow densities.

In [ ]:
def prepare_PRESEL_class(df, label, max_events, random_state):
    """Build one class with unit-normalized weights for BCE training."""
    n_events = min(len(df), int(max_events))
    sample = df.sample(n=n_events, random_state=random_state).copy()
    sample["PRESEL_label"] = float(label)
    sample["PRESEL_weight"] = sample["weight"] / sample["weight"].sum()
    return sample


PRESEL_signal_bce = prepare_PRESEL_class(
    PRESEL_signal_train,
    label=1,
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    random_state=SEED + 11,
)
PRESEL_background_bce = prepare_PRESEL_class(
    PRESEL_background_train,
    label=0,
    max_events=PRESEL_MAX_TRAIN_EVENTS_PER_CLASS,
    random_state=SEED + 22,
)
PRESEL_training_dataframe = pd.concat(
    [PRESEL_signal_bce, PRESEL_background_bce],
    ignore_index=True,
).sample(frac=1.0, random_state=SEED, ignore_index=True)

PRESEL_trainer = density_ratio_trainer(
    dataset=PRESEL_training_dataframe,
    weights=PRESEL_training_dataframe["PRESEL_weight"].to_numpy(),
    training_labels=PRESEL_training_dataframe["PRESEL_label"].to_numpy(),
    features=FEATURES,
    features_scaling=FEATURES,
    sample_name=["signal", "background"],
    output_name="PRESEL",
    path_to_figures=f"{PRESEL_PLOT_DIR}/",
    path_to_models=f"{PRESEL_MODEL_DIR}/",
)

PRESEL_trainer.train(
    hidden_layers=PRESEL_HIDDEN_LAYERS,
    neurons=PRESEL_NEURONS,
    number_of_epochs=PRESEL_N_EPOCHS,
    batch_size=PRESEL_BATCH_SIZE,
    learning_rate=PRESEL_LEARNING_RATE,
    scalerType="MinMax",
    ensemble_index=0,
    verbose=1,
    rnd_seed=SEED,
    holdout_split=PRESEL_HOLDOUT_FRACTION,
    validation_split=PRESEL_VALIDATION_FRACTION,
    callback_patience=PRESEL_PATIENCE,
    num_workers=0,
    load_trained_models=PRESEL_LOAD_IF_AVAILABLE,
    calibration=False,
)

### Choose the preselection cut

We now require a sufficiently large learned ratio $r_{\rm PRESEL}(x)$. The threshold is chosen on the future flow-training sample—not on the final evaluation sample—as the loosest cut that reaches the target $B/S$. This keeps as much signal as possible while reducing the background burden to roughly $B/S=100$.

After the cut, each flow learns the **conditional** density $p(x\mid\mathrm{pass},S)$ or $p(x\mid\mathrm{pass},B)$. Consequently, the extended likelihood below must use the corresponding post-selection signal and background yields.

In [ ]:
def choose_PRESEL_ratio_cut(
    signal_ratio,
    background_ratio,
    signal_weight,
    background_weight,
    target_background_to_signal,
):
    """Loosest weighted ratio cut satisfying the requested B/S target."""
    ratio = np.concatenate([signal_ratio, background_ratio]).astype(float)
    signal_w = np.concatenate(
        [np.asarray(signal_weight, dtype=float), np.zeros(len(background_ratio))]
    )
    background_w = np.concatenate(
        [np.zeros(len(signal_ratio)), np.asarray(background_weight, dtype=float)]
    )

    # Group equal classifier outputs before accumulating, so the returned >=
    # cut has exactly the yield used to choose it.
    unique_ratio, inverse = np.unique(ratio, return_inverse=True)
    signal_by_ratio = np.bincount(inverse, weights=signal_w)
    background_by_ratio = np.bincount(inverse, weights=background_w)

    unique_ratio = unique_ratio[::-1]
    cumulative_signal = np.cumsum(signal_by_ratio[::-1])
    cumulative_background = np.cumsum(background_by_ratio[::-1])
    background_to_signal = np.divide(
        cumulative_background,
        cumulative_signal,
        out=np.full_like(cumulative_background, np.inf),
        where=cumulative_signal > 0.0,
    )

    valid = np.flatnonzero(
        (cumulative_signal > 0.0)
        & (background_to_signal <= float(target_background_to_signal))
    )
    if len(valid) == 0:
        raise RuntimeError(
            "The PRESEL classifier cannot reach the requested B/S target. "
            "Train it longer or relax PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO."
        )

    # Among valid cuts, maximize retained signal yield.
    best = valid[np.argmax(cumulative_signal[valid])]
    return float(unique_ratio[best])


def evaluate_PRESEL_ratio(df):
    ratio = predict_with_model(
        df[FEATURES].astype("float32"),
        scaler=PRESEL_trainer.scaler,
        model=PRESEL_trainer.model_NN,
    )
    return np.asarray(ratio, dtype=float).reshape(-1)


PRESEL_signal_ratio_train = evaluate_PRESEL_ratio(signal_train_unselected)
PRESEL_background_ratio_train = evaluate_PRESEL_ratio(background_train_unselected)
PRESEL_signal_ratio_eval = evaluate_PRESEL_ratio(signal_eval_unselected)
PRESEL_background_ratio_eval = evaluate_PRESEL_ratio(background_eval_unselected)

PRESEL_RATIO_CUT = choose_PRESEL_ratio_cut(
    PRESEL_signal_ratio_train,
    PRESEL_background_ratio_train,
    signal_train_unselected["weight"],
    background_train_unselected["weight"],
    PRESEL_TARGET_BACKGROUND_TO_SIGNAL_RATIO,
)

PRESEL_signal_mask_train = PRESEL_signal_ratio_train >= PRESEL_RATIO_CUT
PRESEL_background_mask_train = PRESEL_background_ratio_train >= PRESEL_RATIO_CUT
PRESEL_signal_mask_eval = PRESEL_signal_ratio_eval >= PRESEL_RATIO_CUT
PRESEL_background_mask_eval = PRESEL_background_ratio_eval >= PRESEL_RATIO_CUT

signal_train = signal_train_unselected.loc[PRESEL_signal_mask_train].reset_index(drop=True)
background_train = background_train_unselected.loc[PRESEL_background_mask_train].reset_index(drop=True)
signal_eval = signal_eval_unselected.loc[PRESEL_signal_mask_eval].reset_index(drop=True)
background_eval = background_eval_unselected.loc[PRESEL_background_mask_eval].reset_index(drop=True)

# Nominal post-selection yields and efficiencies are determined without using
# the final evaluation sample. Rescale the selected evaluation weights to the
# same nominal yields so its Asimov representation is consistent.
PRESEL_SELECTED_YIELD = {
    "signal": float(signal_train["weight"].sum()),
    "background": float(background_train["weight"].sum()),
}
PRESEL_EFFICIENCY = {
    sample_name: PRESEL_SELECTED_YIELD[sample_name] / PRESEL_INCLUSIVE_YIELD[sample_name]
    for sample_name in ["signal", "background"]
}
PRESEL_EVAL_YIELD_BEFORE_RESCALE = {
    "signal": float(signal_eval["weight"].sum()),
    "background": float(background_eval["weight"].sum()),
}

for sample_name, df_eval in [("signal", signal_eval), ("background", background_eval)]:
    if len(df_eval) == 0 or df_eval["weight"].sum() <= 0.0:
        raise RuntimeError(f"The PRESEL cut removed every {sample_name} evaluation event.")
    df_eval["weight"] *= PRESEL_SELECTED_YIELD[sample_name] / df_eval["weight"].sum()

TOTAL_YIELD = PRESEL_SELECTED_YIELD.copy()

PRESEL_train_background_to_signal = (
    PRESEL_SELECTED_YIELD["background"] / PRESEL_SELECTED_YIELD["signal"]
)
PRESEL_eval_background_to_signal = (
    PRESEL_EVAL_YIELD_BEFORE_RESCALE["background"]
    / PRESEL_EVAL_YIELD_BEFORE_RESCALE["signal"]
)

print(f"PRESEL ratio cut: r >= {PRESEL_RATIO_CUT:.5g}")
print(
    "Signal efficiency = "
    f"{PRESEL_EFFICIENCY['signal']:.3%}; background efficiency = "
    f"{PRESEL_EFFICIENCY['background']:.3%}"
)
print(
    f"Post-selection B/S: flow training = {PRESEL_train_background_to_signal:.2f}, "
    f"independent evaluation = {PRESEL_eval_background_to_signal:.2f}"
)
print(
    f"Flow training after PRESEL: {len(signal_train):,} signal + "
    f"{len(background_train):,} background events"
)
print(
    f"Fit/eval after PRESEL:      {len(signal_eval):,} signal + "
    f"{len(background_eval):,} background events"
)
print("Post-selection yields:", TOTAL_YIELD)

## Choose and train the final normalizing flows

The implementation is kept in `utils_nf.py` so that this workshop notebook can focus on the statistical ideas. The boolean `USE_QUADRATIC_SPLINE` above selects one of two **discrete normalizing flows**:

- **RealNVP:** affine coupling transforms with alternating binary masks;
- **quadratic spline:** piecewise rational-quadratic coupling transforms, which can represent more flexible monotonic maps within each coupling layer.

Both are trained only on events passing PRESEL, by maximizing the exact log likelihood

$$
\log \hat p(x) = \log p_Z\big(f(x)\big)
+ \log\left|\det\frac{\partial f}{\partial x}\right|.
$$

For numerical stability, each sample is standardized before training. The utility functions include the standardization Jacobian when evaluating a density, so `flow_log_prob_x` always returns $\log \hat p(x)$ in the original `x` coordinates.

### Train the signal and background flows

The architecture and optimizer settings are passed explicitly from the notebook to `train_flow`. The inputs here are the post-selection signal and background samples. Checkpoints are saved under `models_flows_preselected/`; RealNVP and spline models use different filenames, so students can train and compare both choices safely.

In [ ]:
flows = {}
for sample_name, sample_train, flow_seed in [
    ("background", background_train, 101),
    ("signal", signal_train, 202),
]:
    flows[sample_name] = train_flow(
        sample_name,
        sample_train,
        features=FEATURES,
        model_dir=FLOW_MODEL_DIR,
        model_config=MODEL_CONFIG,
        training_config=TRAINING_CONFIG,
        device=device,
        max_train_events=MAX_TRAIN_EVENTS[sample_name],
        load_if_available=LOAD_IF_AVAILABLE,
        seed=flow_seed,
    )

## Validation 1: feature and correlation closure

A flow must reproduce not only every one-dimensional feature distribution, but also the dependence among features. The pair plot below summarizes both:

- **diagonal:** overlaid one-dimensional densities for held-out MC and flow-generated events;
- **lower triangle:** shared-range two-dimensional density contours enclosing approximately 68% and 95% of each sample;
- **upper triangle:** the Pearson correlations $\rho_\mathrm{MC}$ and $\rho_\mathrm{flow}$, together with $\Delta\rho = \rho_\mathrm{flow}-\rho_\mathrm{MC}$.

The contours are particularly important: matching all diagonal projections does not guarantee that the flow has learned the joint five-dimensional structure.

In [ ]:
def sample_flow_pair_closure(df_eval, flow_pack, n_plot=50_000):
    """Prepare held-out and generated event arrays for the pair plot."""
    n_plot = min(int(n_plot), len(df_eval))
    mc = df_eval.sample(n=n_plot, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    generated = flow_sample_x(flow_pack, n_plot)
    return mc, generated


background_pair_mc, background_pair_generated = sample_flow_pair_closure(
    background_eval, flows["background"]
)
fig = plot_flow_pair_closure(
    "background",
    background_pair_mc,
    background_pair_generated,
    FEATURES,
)
plt.show()

signal_pair_mc, signal_pair_generated = sample_flow_pair_closure(
    signal_eval, flows["signal"]
)
fig = plot_flow_pair_closure(
    "signal",
    signal_pair_mc,
    signal_pair_generated,
    FEATURES,
)
plt.show()

## Validation 2: closure as a function of the learned log density

The five reconstructed variables can also be compressed to the scalar learned log density

$$
\ell(x) = \log \hat p_\mathrm{flow}(x).
$$

Below, we evaluate the **same trained flow** on two sets of events: held-out MC events and events sampled from the flow. If the learned density reproduces the data distribution, the two distributions of $\ell(x)$ should agree.

The generated reference uses many more events by default than the held-out MC points. Its finite-sampling fluctuations are therefore small, and the visible statistical uncertainty is dominated by the finite MC evaluation sample. The lower panel shows the bin-by-bin ratio $\mathrm{MC}/\mathrm{flow}$; its error bars include the finite counts in both histograms, although the high-statistics flow contribution is small. This does **not** remove uncertainty or bias in the trained flow itself; it only makes the numerical sampling of that fixed flow very precise.

### CDF-versus-CDF closure

The same comparison can be displayed as a probability–probability (P–P) plot. For common thresholds in $\ell=\log \hat p(x)$, we plot the empirical MC CDF against the flow CDF,

$$
F_\mathrm{MC}(\ell) \quad\text{versus}\quad F_\mathrm{flow}(\ell).
$$

Perfect closure therefore follows the diagonal $F_\mathrm{MC}=F_\mathrm{flow}$. The lower panel shows the ratio $F_\mathrm{MC}/F_\mathrm{flow}$; the scan starts at $F_\mathrm{flow}=10^{-3}$ to avoid division by zero. The blue bands are the approximate pointwise $\pm1\sigma$ finite-sample expectations; neighboring CDF points are correlated, and the bands do not include flow-training uncertainty.

This remains a one-dimensional projection. It is a useful and sensitive diagnostic, but agreement here alone does not prove complete five-dimensional closure because different regions of feature space can have the same value of $\log \hat p(x)$.

In [ ]:
def evaluate_log_prob_closure(
    df_eval,
    flow_pack,
    n_mc=50_000,
    n_flow_samples=1_000_000,
):
    """Prepare learned log densities for held-out and generated events."""
    n_mc = min(int(n_mc), len(df_eval))
    mc_events = df_eval.sample(n=n_mc, random_state=SEED)[FEATURES]
    log_p_mc = flow_log_prob_x(flow_pack, mc_events)

    # A much larger generated reference makes its finite-sampling noise small
    # compared with that of the held-out MC sample.
    generated_events = flow_sample_x(flow_pack, int(n_flow_samples))
    log_p_generated = flow_log_prob_x(flow_pack, generated_events)
    del generated_events

    log_p_mc = log_p_mc[np.isfinite(log_p_mc)]
    log_p_generated = log_p_generated[np.isfinite(log_p_generated)]
    if len(log_p_mc) == 0 or len(log_p_generated) == 0:
        raise ValueError("The flow returned no finite log-probability values.")
    return log_p_mc, log_p_generated


background_log_p_mc, background_log_p_generated = evaluate_log_prob_closure(
    background_eval, flows["background"]
)
fig = plot_log_prob_closure(
    "background", background_log_p_mc, background_log_p_generated
)
plt.show()
cdf_fig = plot_log_prob_cdf_closure(
    "background", background_log_p_mc, background_log_p_generated, color="C0"
)
plt.show()

signal_log_p_mc, signal_log_p_generated = evaluate_log_prob_closure(
    signal_eval, flows["signal"]
)
fig = plot_log_prob_closure(
    "signal", signal_log_p_mc, signal_log_p_generated
)
plt.show()
cdf_fig = plot_log_prob_cdf_closure(
    "signal", signal_log_p_mc, signal_log_p_generated, color="C4"
)
plt.show()

## Validation 3: compare with the analytic smeared truth

For the generator, the truth-level variables are Gaussian mixtures in $y$.  The reco variables are generated as

$$
x = D y + \epsilon, \qquad \epsilon_i \sim \mathcal{N}(0, \sigma_{\mathrm{res},i}^2),
$$

where $D = \mathrm{diag}(\mathrm{scale})$.  Therefore each Gaussian component remains Gaussian after smearing:

$$
\mu_x = D\mu_y, \qquad \Sigma_x = D\Sigma_yD^T + \mathrm{diag}(\sigma_\mathrm{res}^2).
$$

This analytic truth is only available because the tutorial toy model is simple.  It is useful here as a clean closure check of the flow density itself.

Because the flows are trained after PRESEL, this comparison uses the analytic density conditional on passing the cut. On selected events this is $p(x\mid\mathrm{pass})=p(x)/\epsilon_{\rm PRESEL}$, where the efficiency is estimated from the independent flow-training pool.

In [ ]:
def reco_components_from_truth_components(components):
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    response_cov = np.diag(resolution ** 2)

    reco_components = []
    for frac, mean_y, cov_y in components:
        mean_x = scale * np.asarray(mean_y, dtype=float)
        cov_x = D @ np.asarray(cov_y, dtype=float) @ D.T + response_cov
        reco_components.append((frac, mean_x, cov_x))
    return reco_components


def mixture_log_density(x, components):
    x = np.asarray(x, dtype=float)
    fracs = np.asarray([c[0] for c in components], dtype=float)
    fracs = fracs / fracs.sum()

    terms = []
    for f, (_, mean, cov) in zip(fracs, components):
        terms.append(
            np.log(f) + multivariate_normal(mean=mean, cov=cov, allow_singular=False).logpdf(x)
        )
    return logsumexp(np.vstack(terms), axis=0)


TRUTH_RECO_COMPONENTS = {
    "background": reco_components_from_truth_components(background_components()),
    "signal": reco_components_from_truth_components(signal_components()),
}


def selected_mixture_log_density(x, sample_name):
    """Analytic reco density conditional on passing the PRESEL cut."""
    return (
        mixture_log_density(x, TRUTH_RECO_COMPONENTS[sample_name])
        - np.log(PRESEL_EFFICIENCY[sample_name])
    )

In [ ]:
def evaluate_log_density_truth(sample_name, df_eval, flow_pack, n_points=25_000):
    """Prepare analytic and learned log-density arrays."""
    n_points = min(n_points, len(df_eval))
    x = df_eval.sample(n=n_points, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    log_p_flow = flow_log_prob_x(flow_pack, x)
    log_p_truth = selected_mixture_log_density(x, sample_name)
    delta = log_p_flow - log_p_truth

    corr = np.corrcoef(log_p_truth, log_p_flow)[0, 1]
    rmse = np.sqrt(np.mean(delta**2))
    bias = np.mean(delta)
    print(f"{sample_name}: corr(log p) = {corr:.4f}, RMSE = {rmse:.4f}, bias = {bias:.4f}")
    return log_p_truth, log_p_flow


background_log_p_truth, background_log_p_flow = evaluate_log_density_truth(
    "background", background_eval, flows["background"]
)
fig = plot_log_density_truth_scatter(
    "background", background_log_p_truth, background_log_p_flow
)
plt.show()

signal_log_p_truth, signal_log_p_flow = evaluate_log_density_truth(
    "signal", signal_eval, flows["signal"]
)
fig = plot_log_density_truth_scatter(
    "signal", signal_log_p_truth, signal_log_p_flow
)
plt.show()

In [ ]:
def build_log_density_truth_calibration(
    sample_name,
    df_eval,
    flow_pack,
    n_points=100_000,
    n_bins=25,
    binning="quantile",
):
    """Evaluate probabilities and aggregate their residuals in truth bins."""
    n_points = min(n_points, len(df_eval))
    x = df_eval.sample(n=n_points, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    log_p_flow = flow_log_prob_x(flow_pack, x)
    log_p_truth = selected_mixture_log_density(x, sample_name)
    delta = log_p_flow - log_p_truth

    if binning == "quantile":
        edges = np.unique(np.quantile(log_p_truth, np.linspace(0.0, 1.0, n_bins + 1)))
    elif binning == "uniform":
        edges = np.linspace(log_p_truth.min(), log_p_truth.max(), n_bins + 1)
    else:
        edges = np.asarray(binning, dtype=float)
    if len(edges) < 3:
        raise ValueError("Need at least two non-empty bins for the calibration plot.")

    eps = 1.0e-9 * max(1.0, float(edges[-1] - edges[0]))
    edges = edges.astype(float).copy()
    edges[0] -= eps
    edges[-1] += eps
    bin_index = np.searchsorted(edges, log_p_truth, side="right") - 1
    bin_index = np.clip(bin_index, 0, len(edges) - 2)

    rows = []
    for i in range(len(edges) - 1):
        mask = bin_index == i
        count = int(mask.sum())
        if count == 0:
            rows.append(
                {
                    "bin": i,
                    "truth_lo": edges[i],
                    "truth_hi": edges[i + 1],
                    "count": 0,
                    "truth_mean": np.nan,
                    "flow_mean": np.nan,
                    "flow_sem": np.nan,
                    "delta_mean": np.nan,
                    "delta_sem": np.nan,
                }
            )
            continue

        flow_std = log_p_flow[mask].std(ddof=1) if count > 1 else 0.0
        delta_std = delta[mask].std(ddof=1) if count > 1 else 0.0
        rows.append(
            {
                "bin": i,
                "truth_lo": edges[i],
                "truth_hi": edges[i + 1],
                "count": count,
                "truth_mean": log_p_truth[mask].mean(),
                "flow_mean": log_p_flow[mask].mean(),
                "flow_sem": flow_std / np.sqrt(count),
                "delta_mean": delta[mask].mean(),
                "delta_sem": delta_std / np.sqrt(count),
            }
        )

    calibration = pd.DataFrame(rows)
    valid = calibration["count"] > 0
    binned_bias = np.average(
        calibration.loc[valid, "delta_mean"],
        weights=calibration.loc[valid, "count"],
    )
    binned_rms = np.sqrt(
        np.average(
            calibration.loc[valid, "delta_mean"] ** 2,
            weights=calibration.loc[valid, "count"],
        )
    )
    max_abs_bias = np.nanmax(np.abs(calibration.loc[valid, "delta_mean"]))
    print(
        f"{sample_name}: binned bias = {binned_bias:.4f}, "
        f"binned RMS bias = {binned_rms:.4f}, "
        f"max |bin bias| = {max_abs_bias:.4f}"
    )
    return calibration, edges, log_p_truth, log_p_flow

In [ ]:
(
    background_logp_calibration,
    background_logp_edges,
    background_logp_truth_binned,
    background_logp_flow_binned,
) = build_log_density_truth_calibration(
    "background", background_eval, flows["background"]
)
fig = plot_log_density_truth_binned(
    "background",
    background_logp_calibration,
    background_logp_edges,
    background_logp_truth_binned,
    background_logp_flow_binned,
)
plt.show()

(
    signal_logp_calibration,
    signal_logp_edges,
    signal_logp_truth_binned,
    signal_logp_flow_binned,
) = build_log_density_truth_calibration(
    "signal", signal_eval, flows["signal"]
)
fig = plot_log_density_truth_binned(
    "signal",
    signal_logp_calibration,
    signal_logp_edges,
    signal_logp_truth_binned,
    signal_logp_flow_binned,
)
plt.show()

## Direct likelihood from the learned densities

Because the flows estimate the post-selection densities $p_\mathrm{sig}(x\mid\mathrm{pass})$ and $p_\mathrm{bkg}(x\mid\mathrm{pass})$ directly, we do **not** need to choose a reference density and we do **not** need to calculate ratios.

Using the post-selection yields $\lambda_\mathrm{sig}$ and $\lambda_\mathrm{bkg}$, define the event probability

$$
p(x \mid \mu) = \frac{1}{\mu\,\lambda_\mathrm{sig}+ \lambda_\mathrm{bkg} }\left( \mu\,\lambda_\mathrm{sig}\,\hat p_\mathrm{sig}(x)
+ \lambda_\mathrm{bkg}\,\hat p_\mathrm{bkg}(x)\right).
$$

Dropping constants independent of $\mu$,

$$
-2\log L(\mu)
= -2 \sum_i w_i \log p(x_i\mid\mu)
- 2\log \mathrm{Poisson}(N_\mathrm{data};\mu\lambda_\mathrm{sig}+\lambda_\mathrm{bkg}).
$$

The code below stores `p_sig` and `p_bkg` as the yield-weighted terms
$\lambda_\mathrm{sig}\hat p_\mathrm{sig}(x_i)$ and
$\lambda_\mathrm{bkg}\hat p_\mathrm{bkg}(x_i)$, so the core likelihood line is just

```python
prob = mu * p_sig + p_bkg
```

In [ ]:
asimov_dataset = pd.concat([background_eval, signal_eval], ignore_index=True).astype("float32").copy()
weights_asimov = np.asarray(asimov_dataset["weight"], dtype=np.float64)
X_asimov = asimov_dataset[FEATURES].to_numpy(dtype=np.float32)

lam_sig = float(TOTAL_YIELD["signal"])
lam_bkg = float(TOTAL_YIELD["background"])

log_p_sig_flow = flow_log_prob_x(flows["signal"], X_asimov)
log_p_bkg_flow = flow_log_prob_x(flows["background"], X_asimov)

In [ ]:
# Flow-based intensities
p_sig_flow_np = lam_sig * np.exp(np.clip(log_p_sig_flow, -745.0, 50.0))
p_bkg_flow_np = lam_bkg * np.exp(np.clip(log_p_bkg_flow, -745.0, 50.0))

# Truth-based intensities
log_p_sig_truth = selected_mixture_log_density(X_asimov, "signal")
log_p_bkg_truth = selected_mixture_log_density(X_asimov, "background")

p_sig_truth_np = lam_sig * np.exp(np.clip(log_p_sig_truth, -745.0, 50.0))
p_bkg_truth_np = lam_bkg * np.exp(np.clip(log_p_bkg_truth, -745.0, 50.0))

## Fit $\mu$ and construct the profile-likelihood-ratio test statistic

The normalizing flows provide the event densities, while JAX provides the likelihood gradient. Minimizing $\mathrm{NLL}=-2\log L$ gives the maximum-likelihood estimate $\hat\mu$. To test a fixed value of $\mu$, we use the **profile likelihood ratio**

$$
t_\mu = -2\log\lambda(\mu)
= -2\log\frac{L(\mu,\hat{\hat{\boldsymbol\theta}}_\mu)}{L(\hat\mu,\hat{\boldsymbol\theta})}
= \mathrm{NLL}(\mu,\hat{\hat{\boldsymbol\theta}}_\mu)-\mathrm{NLL}(\hat\mu,\hat{\boldsymbol\theta}).
$$

Here $\boldsymbol\theta$ denotes nuisance parameters, and $\hat{\hat{\boldsymbol\theta}}_\mu$ is their conditional best fit at fixed $\mu$. This exercise has no nuisance parameters, so the profiling is trivial. The scan below constructs the test-statistic curve $t_\mu$, whose minimum is $t_{\hat\mu}=0$.

In [ ]:
p_sig = jnp.asarray(p_sig_flow_np, dtype=jnp.float64)
p_bkg = jnp.asarray(p_bkg_flow_np, dtype=jnp.float64)
weights = jnp.asarray(weights_asimov, dtype=jnp.float64)
EPS = 1.0e-300


def nll_manual(params):
    mu = params[0]
    prob = mu * p_sig + p_bkg
    n_expected = mu * lam_sig + lam_bkg

    # The maximum protects the log.  The final where() enforces the physical
    # region and keeps the optimizer away from invalid negative intensities.
    nll = -2.0 * jnp.sum(weights * jnp.log(jnp.maximum(prob, EPS))) + 2.0 * n_expected
    invalid = (mu < 0.0) | (n_expected <= 0.0) | jnp.any(prob <= 0.0)
    return jnp.where(invalid, 1.0e30, nll)

inf_manual = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_manual),
    initial_values=[1.0],
    list_parameters=["mu"],
    num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_manual)),  # analytic gradient, for free
)

print("\n" + "=" * 40)
print(" DIRECT FLOW-DENSITY FIT RESULTS ")
print("=" * 40 + "\n")
inf_manual.perform_fit(freeze_params=[])
mu_min = inf_manual.pulls_global_fit

In [ ]:
SCAN_RANGE = (0.0, 10.0)
scan_flow, tmu_flow = inf_manual.perform_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=50,
)

fig = plot_profile_scan(scan_flow, tmu_flow, label="Flow densities")
plt.show()

In [ ]:
def make_direct_density_nll(p_sig_np, p_bkg_np, weights_np, lam_sig_value, lam_bkg_value):
    p_sig_j = jnp.asarray(p_sig_np, dtype=jnp.float64)
    p_bkg_j = jnp.asarray(p_bkg_np, dtype=jnp.float64)
    weights_j = jnp.asarray(weights_np, dtype=jnp.float64)
    lam_sig_value = float(lam_sig_value)
    lam_bkg_value = float(lam_bkg_value)

    def _nll(params):
        mu = params[0]
        prob = mu * p_sig_j + p_bkg_j
        n_expected = mu * lam_sig_value + lam_bkg_value
        nll = -2.0 * jnp.sum(weights_j * jnp.log(jnp.maximum(prob, EPS))) + 2.0 * n_expected
        invalid = (mu < 0.0) | (n_expected <= 0.0) | jnp.any(prob <= 0.0)
        return jnp.where(invalid, 1.0e30 + 1.0e12 * (mu - 1.0) ** 2, nll)

    return _nll


log_p_sig_truth = selected_mixture_log_density(X_asimov, "signal")
log_p_bkg_truth = selected_mixture_log_density(X_asimov, "background")

p_sig_truth_np = lam_sig * np.exp(np.clip(log_p_sig_truth, -745.0, 50.0))
p_bkg_truth_np = lam_bkg * np.exp(np.clip(log_p_bkg_truth, -745.0, 50.0))

nll_truth = make_direct_density_nll(
    p_sig_truth_np,
    p_bkg_truth_np,
    weights_asimov,
    lam_sig,
    lam_bkg,
)

inf_truth = nsbi_common_utils.inference.inference(
    model_nll=jax.jit(nll_truth),
    initial_values=[1.0],
    list_parameters=["mu"],
    num_unconstrained_params=1,
    model_grad=jax.jit(jax.grad(nll_truth)),
)

print("\n" + "=" * 40)
print(" ANALYTIC RECO-DENSITY FIT RESULTS ")
print("=" * 40 + "\n")
inf_truth.perform_fit(freeze_params=[])
mu_min_truth = inf_truth.pulls_global_fit

scan_truth, tmu_truth = inf_truth.perform_profile_scan(
    parameter_name="mu",
    freeze_params=[],
    bound_range=SCAN_RANGE,
    fit_strategy=0,
    size=50,
)

fig = plot_profile_scan_comparison(
    scan_flow,
    tmu_flow,
    mu_min,
    scan_truth,
    tmu_truth,
    mu_min_truth,
)
plt.show()

## Sampling distributions from pseudo-experiments

For each hypothesis $\mu=0$ and $\mu=1$, a toy first draws its total event count from $\mathrm{Poisson}(\mu\lambda_S+\lambda_B)$. Each event is then labeled signal or background with the corresponding mixture probability and sampled from that post-selection flow. Finally, the toy is fitted with the same JAX likelihood and Minuit interface used above, giving one value of $\hat\mu$ and $t_\mu$.

Repeating this construction produces the sampling distributions below, which we compare with the asymptotic Wilks/Cowan predictions.

In [ ]:
# =============================================================================
# Toy distributions of t_mu and mu_hat
# =============================================================================
# These toys are generated from the trained flows and fitted with the same
# flows. They test the statistical construction and the Wilks/Cowan limits
# under the learned model; they do not test misspecification of that model.

from contextlib import nullcontext, redirect_stdout
from io import StringIO

N_TOYS = 200                 # use 20 for a quick test; increase for smooth plots
TOY_HYPOTHESES = (0.0, 1.0)
TOY_SEED = 314159
TOY_FLOW_BATCH_SIZE = 65_536
TOY_PLOT_BINS = 35
TOY_VERBOSE_FITS = False      # True prints the Minuit table for every toy
TOY_PADDING_SIGMAS = 8.0      # common JAX shape; Poisson overflow is negligible

TOY_MAX_EXPECTED_EVENTS = max(
    mu * lam_sig + lam_bkg for mu in TOY_HYPOTHESES
)
TOY_PAD_SIZE = int(
    np.ceil(
        TOY_MAX_EXPECTED_EVENTS
        + TOY_PADDING_SIGMAS * np.sqrt(TOY_MAX_EXPECTED_EVENTS)
        + 32
    )
)
print(f"JAX toy padding size: {TOY_PAD_SIZE:,} events")


def generate_flow_toy(
    mu_true,
    flow_models,
    lam_sig_value,
    lam_bkg_value,
    rng,
    batch_size=65_536,
):
    """Generate one extended toy dataset from the signal/background flows."""
    mu_true = float(mu_true)
    if mu_true < 0.0:
        raise ValueError("mu_true must be non-negative.")

    n_expected = mu_true * lam_sig_value + lam_bkg_value
    n_total = int(rng.poisson(n_expected))
    signal_fraction = (
        mu_true * lam_sig_value / n_expected if n_expected > 0.0 else 0.0
    )
    n_signal = int(rng.binomial(n_total, signal_fraction))
    n_background = n_total - n_signal

    samples = []
    if n_background:
        samples.append(
            flow_sample_x(
                flow_models["background"],
                n_background,
                batch_size=batch_size,
            )
        )
    if n_signal:
        samples.append(
            flow_sample_x(
                flow_models["signal"],
                n_signal,
                batch_size=batch_size,
            )
        )

    if samples:
        x_toy = np.concatenate(samples, axis=0).astype(np.float32, copy=False)
    else:
        x_toy = np.empty((0, len(FEATURES)), dtype=np.float32)

    # Event order is irrelevant for an unbinned likelihood, so no expensive
    # million-row shuffle is needed.
    return x_toy, n_total, n_signal, n_background



def _toy_relative_nll(params, q_padded, lam_sig_value, lam_bkg_value):
    """-2 log L up to background-only constants, for one padded toy."""
    mu = params[0]
    event_factor = 1.0 + mu * q_padded
    n_expected = mu * lam_sig_value + lam_bkg_value
    nll = 2.0 * (
        mu * lam_sig_value
        - jnp.sum(jnp.log(jnp.maximum(event_factor, EPS)))
    )
    invalid = (n_expected <= 0.0) | jnp.any(event_factor <= 0.0)
    return jnp.where(invalid, 1.0e30 + 1.0e12 * mu**2, nll)


# All toys have the same padded shape, so JAX compiles these kernels only once.
TOY_NLL_KERNEL = jax.jit(_toy_relative_nll)
TOY_GRAD_KERNEL = jax.jit(jax.grad(_toy_relative_nll, argnums=0))


def fit_flow_toy(
    x_toy,
    test_mu,
    flow_models,
    lam_sig_value,
    lam_bkg_value,
    batch_size=65_536,
):
    """Fit one toy with the common JAX/Minuit inference interface."""
    if len(x_toy) == 0:
        return 0.0, 2.0 * float(test_mu) * lam_sig_value, 0.0
    if len(x_toy) > TOY_PAD_SIZE:
        raise RuntimeError(
            f"Toy has {len(x_toy):,} events, above TOY_PAD_SIZE={TOY_PAD_SIZE:,}. "
            "Increase TOY_PADDING_SIGMAS."
        )

    log_p_sig_toy = flow_log_prob_x(
        flow_models["signal"], x_toy, batch_size=batch_size
    ).astype(np.float64, copy=False)
    log_p_bkg_toy = flow_log_prob_x(
        flow_models["background"], x_toy, batch_size=batch_size
    ).astype(np.float64, copy=False)

    # q_i is the signal/background intensity ratio. Background-only constants
    # cancel, reducing the JAX input from two density arrays to one ratio array.
    log_q = (
        np.log(lam_sig_value / lam_bkg_value)
        + log_p_sig_toy
        - log_p_bkg_toy
    )
    q = np.exp(np.clip(log_q, -80.0, 80.0))
    del log_p_sig_toy, log_p_bkg_toy, log_q

    q_padded = np.zeros(TOY_PAD_SIZE, dtype=np.float64)
    q_padded[: len(q)] = q
    q_jax = jnp.asarray(q_padded)
    del q_padded

    def toy_nll(params):
        return TOY_NLL_KERNEL(params, q_jax, lam_sig_value, lam_bkg_value)

    def toy_grad(params):
        return TOY_GRAD_KERNEL(params, q_jax, lam_sig_value, lam_bkg_value)

    toy_inference = nsbi_common_utils.inference.inference(
        model_nll=toy_nll,
        initial_values=[max(0.1, float(test_mu))],
        list_parameters=["mu"],
        num_unconstrained_params=1,
        model_grad=toy_grad,
    )

    # Minuit first finds the unconstrained one-dimensional optimum. Since this
    # likelihood is convex, imposing mu >= 0 is simply a projection to zero.
    fit_output = nullcontext() if TOY_VERBOSE_FITS else redirect_stdout(StringIO())
    with fit_output:
        toy_inference.perform_fit(fit_strategy=0, freeze_params=[])
    mu_unconstrained = float(np.asarray(toy_inference.pulls_global_fit).reshape(-1)[0])
    mu_hat = max(0.0, mu_unconstrained)

    # Evaluate the likelihood-ratio difference in NumPy after the Minuit fit.
    # This avoids retaining a JAX object from each toy.
    test_mu = float(test_mu)
    t_mu = 2.0 * (
        (test_mu - mu_hat) * lam_sig_value
        - np.sum(np.log1p(test_mu * q) - np.log1p(mu_hat * q))
    )
    t_mu = max(0.0, float(t_mu))

    response = q / (1.0 + test_mu * q)
    observed_information = float(np.sum(response**2))
    return mu_hat, t_mu, observed_information


def run_flow_toys(
    mu_true,
    n_toys,
    flow_models,
    lam_sig_value,
    lam_bkg_value,
    seed,
    batch_size=65_536,
):
    """Generate and fit a sequence of toys while keeping memory bounded."""
    n_toys = int(n_toys)
    if n_toys < 1:
        raise ValueError("n_toys must be positive.")

    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rows = []
    progress_every = max(1, n_toys // 10)
    for toy_index in range(n_toys):
        x_toy, n_total, n_signal, n_background = generate_flow_toy(
            mu_true,
            flow_models,
            lam_sig_value,
            lam_bkg_value,
            rng,
            batch_size=batch_size,
        )
        mu_hat, t_mu, information = fit_flow_toy(
            x_toy,
            mu_true,
            flow_models,
            lam_sig_value,
            lam_bkg_value,
            batch_size=batch_size,
        )
        rows.append(
            {
                "mu_true": float(mu_true),
                "toy": toy_index,
                "n_events": n_total,
                "n_signal": n_signal,
                "n_background": n_background,
                "mu_hat": mu_hat,
                "t_mu": t_mu,
                "information": information,
            }
        )
        del x_toy

        if (toy_index + 1) % progress_every == 0 or toy_index + 1 == n_toys:
            print(f"mu={mu_true:g}: completed {toy_index + 1}/{n_toys} toys")

    return pd.DataFrame(rows)


def asymptotic_sigmas(toy_results):
    """Estimate the expected Wald sigma from the mean observed information."""
    sigmas = {}
    for mu_true, group in toy_results.groupby("mu_true"):
        mean_information = float(group["information"].mean())
        sigmas[float(mu_true)] = 1.0 / np.sqrt(mean_information)
    return sigmas


toy_results = pd.concat(
    [
        run_flow_toys(
            mu_true,
            N_TOYS,
            flows,
            lam_sig,
            lam_bkg,
            seed=TOY_SEED + int(1000 * mu_true),
            batch_size=TOY_FLOW_BATCH_SIZE,
        )
        for mu_true in TOY_HYPOTHESES
    ],
    ignore_index=True,
)

sigma_by_mu = asymptotic_sigmas(toy_results)
print("Asymptotic sigma_mu estimates:", sigma_by_mu)
print(
    toy_results.groupby("mu_true")[["mu_hat", "t_mu", "n_events"]]
    .agg(["mean", "std"])
    .to_string()
)

fig = plot_t_mu_toys(toy_results, n_bins=TOY_PLOT_BINS)
plt.show()

fig = plot_mu_hat_toys(toy_results, sigma_by_mu, n_bins=TOY_PLOT_BINS)
plt.show()

## Suggested exercises

1. Toggle `USE_QUADRATIC_SPLINE`, retrain the two flows, and compare the validation plots with the RealNVP result.
2. Vary the number of coupling layers, hidden-layer width, and spline bins to study the trade-off between expressivity and training time.
3. Train on truth-level `y*` variables instead of reco-level `x*` variables and compare how detector smearing changes the achievable sensitivity.